# LeetCode 746 — Min Cost Climbing Stairs bằng Python

**Link đề:** https://leetcode.com/problems/min-cost-climbing-stairs/
**Độ khó:** Easy · **Chủ đề:** Array, Dynamic Programming

```python
class Solution:
    def minCostClimbingStairs(self, cost: list[int]) -> int:
        ...
```

## Mục tiêu học
1. Mô hình hóa bài toán: **trạng thái** là gì, **đích** ở đâu (bậc `n`, ngay sau bậc cuối), **chi phí** trả khi nào.
2. Đi từ **đệ quy thử mọi cách** (`O(2^n)`) → thấy **bài toán con lặp lại** → **ghi nhớ** (memoization, `@lru_cache`).
3. Chuyển sang **Quy hoạch động bottom-up** với mảng `dp`, rồi tối ưu bộ nhớ còn **2 biến** `O(1)`.
4. Trace bảng `dp`, chứng minh công thức truy hồi, phân tích độ phức tạp, test đối chiếu.
5. Luyện các biến thể cùng họ: Climbing Stairs (70), House Robber (198), truy vết đường đi.

## 1. Đề bài

Cầu thang có `n` bậc đánh số `0..n-1`; `cost[i]` là phí của bậc `i`.

- Khi **đứng ở bậc `i` và trả `cost[i]`**, bạn được bước lên **1 hoặc 2** bậc.
- Được chọn **bắt đầu ở bậc 0 hoặc bậc 1** (đứng vào bậc xuất phát là miễn phí; chỉ trả khi rời bậc đó).
- **Đỉnh** (top) là vị trí `n` — ngay sau bậc cuối. Đứng ở đỉnh thì không phải trả gì thêm.

Trả về **chi phí nhỏ nhất** để lên tới đỉnh.

**Ví dụ 1:** `cost = [10,15,20]` → `15`. Bắt đầu ở bậc 1, trả 15, bước 2 bậc → đỉnh.

**Ví dụ 2:** `cost = [1,100,1,1,1,100,1,1,100,1]` → `6`. Đi `0 → 2 → 4 → 6 → 7 → 9 → đỉnh`, trả 6 lần phí 1.

**Constraints:** `2 <= len(cost) <= 1000`, `0 <= cost[i] <= 999`.

### Hình dung bằng sơ đồ

```text
vị trí :   0     1     2    | 3 = đỉnh
cost   :  10    15    20    |  (không có phí)

Từ 1: trả 15, nhảy +2 -> 3 (đỉnh).  Tổng 15.
Từ 0: trả 10, nhảy +2 -> 2; trả 20, nhảy +1 -> 3.  Tổng 30.
Từ 0: trả 10, nhảy +1 -> 1; trả 15, nhảy +2 -> 3.  Tổng 25.
```

Câu hỏi mấu chốt: "**muốn tới vị trí `i`, bước cuối cùng xuất phát từ đâu?**" — chỉ có thể từ `i-1` (nhảy 1) hoặc `i-2` (nhảy 2).

In [1]:
cost = [10, 15, 20]
n = len(cost)
paths = {
    "bắt đầu 1, nhảy 2": cost[1],
    "bắt đầu 0, nhảy 2, nhảy 1": cost[0] + cost[2],
    "bắt đầu 0, nhảy 1, nhảy 2": cost[0] + cost[1],
    "bắt đầu 1, nhảy 1, nhảy 1": cost[1] + cost[2],
}
for k, v in paths.items():
    print(f"{k:<28} -> {v}")
print("nhỏ nhất:", min(paths.values()))

bắt đầu 1, nhảy 2            -> 15
bắt đầu 0, nhảy 2, nhảy 1    -> 30
bắt đầu 0, nhảy 1, nhảy 2    -> 25
bắt đầu 1, nhảy 1, nhảy 1    -> 35
nhỏ nhất: 15


## 2. Nền tảng Python

### 2.1. `min` của hai giá trị và `dict.items()`

`min(a, b)` trả giá trị nhỏ hơn. `d.items()` cho các cặp `(key, value)` — đã dùng ở cell trên để in mọi đường đi.

### 2.2. Hàm đệ quy

Hàm gọi lại chính nó với **bài toán nhỏ hơn**, và phải có **điểm dừng** (base case). Ví dụ kinh điển: giai thừa.

In [2]:
def factorial(k):
    if k <= 1:          # base case
        return 1
    return k * factorial(k - 1)

print(factorial(5))

120


### 2.3. `functools.lru_cache` — ghi nhớ kết quả hàm

Decorator `@lru_cache(maxsize=None)` (hoặc `@cache` từ Python 3.9) tự lưu kết quả theo tham số; lần sau gọi với cùng tham số thì trả ngay không tính lại. Đây là **memoization**.

In [3]:
from functools import lru_cache

calls = 0

@lru_cache(maxsize=None)
def slow_square(x):
    global calls
    calls += 1
    return x * x

print(slow_square(7), slow_square(7), slow_square(8))
print("số lần thân hàm thật sự chạy:", calls)   # 2, không phải 3

49 49 64
số lần thân hàm thật sự chạy: 2


`global calls` cho phép hàm **gán lại** biến toàn cục `calls`.

### 2.4. Tạo list kích thước cố định

`[0] * (n + 1)` tạo list `n + 1` số 0 — dùng làm bảng `dp`.

In [4]:
n = 3
dp = [0] * (n + 1)
print(dp, len(dp))

[0, 0, 0, 0] 4


## 3. Định nghĩa trạng thái và công thức truy hồi

Đặt **`f(i)` = chi phí nhỏ nhất để ĐỨNG ở vị trí `i`** (chưa trả phí của bậc `i`).

- **Base case:** `f(0) = 0`, `f(1) = 0` (được chọn đứng xuất phát ở 0 hoặc 1 miễn phí).
- **Truy hồi** (với `i >= 2`): bước cuối đến `i` xuất phát từ `i-1` (trả `cost[i-1]`) hoặc từ `i-2` (trả `cost[i-2]`):

```text
f(i) = min( f(i-1) + cost[i-1],  f(i-2) + cost[i-2] )
```

- **Đáp án:** `f(n)` (đỉnh).

Kiểm tra với `cost = [10, 15, 20]`:
- `f(2) = min(f(1) + 15, f(0) + 10) = min(15, 10) = 10`
- `f(3) = min(f(2) + 20, f(1) + 15) = min(30, 15) = 15` ✅

In [5]:
cost = [10, 15, 20]
f0, f1 = 0, 0
f2 = min(f1 + cost[1], f0 + cost[0])
f3 = min(f2 + cost[2], f1 + cost[1])
print("f(2) =", f2, "| f(3) =", f3)

f(2) = 10 | f(3) = 15


## 4. Cách 1 — Đệ quy thuần (Brute Force)

Viết thẳng công thức. Đúng nhưng mỗi lời gọi tách thành 2 lời gọi → cây đệ quy phình theo cấp số nhân.

In [6]:
def min_cost_recursive(cost):
    def f(i):
        if i <= 1:
            return 0
        return min(f(i - 1) + cost[i - 1], f(i - 2) + cost[i - 2])
    return f(len(cost))

print(min_cost_recursive([10, 15, 20]))
print(min_cost_recursive([1, 100, 1, 1, 1, 100, 1, 1, 100, 1]))

15
6


### Nút thắt: bài toán con lặp lại

Đếm số lần `f` được gọi. Số lời gọi tăng như dãy Fibonacci (~`1.618^n`) — với `n = 1000` là con số vô vọng.

In [7]:
def count_calls(n):
    cnt = 0
    def f(i):
        nonlocal cnt
        cnt += 1
        if i <= 1:
            return 0
        return min(f(i - 1), f(i - 2))
    f(n)
    return cnt

for n in [5, 10, 20, 25]:
    print(f"n = {n:>2}: {count_calls(n):>8,} lời gọi")

n =  5:       15 lời gọi
n = 10:      177 lời gọi
n = 20:   21,891 lời gọi
n = 25:  242,785 lời gọi


Cây gọi của `f(5)`: `f(3)` được tính 2 lần, `f(2)` 3 lần, ... Cùng một câu hỏi "chi phí nhỏ nhất tới bậc `i`" bị trả lời đi trả lời lại.

```text
               f(5)
             /      \
          f(4)       f(3)
         /   \       /  \
      f(3)  f(2)  f(2)  f(1)
      / \    ...   ...
   f(2) f(1)
```

**Insight:** chỉ có `n + 1` câu hỏi khác nhau (`f(0)..f(n)`). Trả lời mỗi câu **một lần** rồi nhớ lại → `O(n)`.

## 5. Cách 2 — Đệ quy có ghi nhớ (Top-down DP)

In [8]:
from functools import lru_cache

def min_cost_memo(cost):
    @lru_cache(maxsize=None)
    def f(i):
        if i <= 1:
            return 0
        return min(f(i - 1) + cost[i - 1], f(i - 2) + cost[i - 2])
    return f(len(cost))

print(min_cost_memo([10, 15, 20]))
print(min_cost_memo([1, 100, 1, 1, 1, 100, 1, 1, 100, 1]))

15
6


Vẫn còn rủi ro: `f(1000)` gọi `f(999)` gọi `f(998)` ... → **độ sâu đệ quy ~1000**, chạm giới hạn mặc định của Python (`sys.getrecursionlimit()` thường 1000). Có thể tự cài memo bằng `dict` và tăng giới hạn, nhưng cách gọn hơn là **lật ngược thứ tự**: tính từ nhỏ đến lớn bằng vòng lặp → Cách 3.

## 6. Cách 3 — Bottom-up DP với mảng `dp`

Tính `dp[0], dp[1], ..., dp[n]` theo thứ tự tăng dần; khi tính `dp[i]`, hai ô `dp[i-1]`, `dp[i-2]` **đã có sẵn**.

In [9]:
def min_cost_table(cost):
    n = len(cost)
    dp = [0] * (n + 1)            # dp[0] = dp[1] = 0
    for i in range(2, n + 1):
        dp[i] = min(dp[i - 1] + cost[i - 1], dp[i - 2] + cost[i - 2])
    return dp[n]

print(min_cost_table([10, 15, 20]))
print(min_cost_table([1, 100, 1, 1, 1, 100, 1, 1, 100, 1]))

15
6


## 7. Trace bảng `dp`

Ví dụ 2: `cost = [1,100,1,1,1,100,1,1,100,1]`, `n = 10`.

| i | từ i-1: dp[i-1] + cost[i-1] | từ i-2: dp[i-2] + cost[i-2] | dp[i] |
|---:|---|---|---:|
| 0 | — | — | 0 |
| 1 | — | — | 0 |
| 2 | 0 + 100 = 100 | 0 + 1 = 1 | 1 |
| 3 | 1 + 1 = 2 | 0 + 100 = 100 | 2 |
| 4 | 2 + 1 = 3 | 1 + 1 = 2 | 2 |
| 5 | 2 + 1 = 3 | 2 + 1 = 3 | 3 |
| 6 | 3 + 100 = 103 | 2 + 1 = 3 | 3 |
| 7 | 3 + 1 = 4 | 3 + 100 = 103 | 4 |
| 8 | 4 + 1 = 5 | 3 + 1 = 4 | 4 |
| 9 | 4 + 100 = 104 | 4 + 1 = 5 | 5 |
| 10 | 5 + 1 = 6 | 4 + 100 = 104 | **6** |

Cell dưới in đúng bảng này và **truy vết** đường đi tối ưu bằng mảng `came_from`.

In [10]:
def trace_min_cost(cost):
    n = len(cost)
    dp = [0] * (n + 1)
    came_from = [None] * (n + 1)
    print(f"{'i':>2} | {'từ i-1':>12} | {'từ i-2':>12} | dp[i]")
    print(f"{0:>2} | {'—':>12} | {'—':>12} | 0")
    print(f"{1:>2} | {'—':>12} | {'—':>12} | 0")
    for i in range(2, n + 1):
        a = dp[i - 1] + cost[i - 1]
        b2 = dp[i - 2] + cost[i - 2]
        if a <= b2:
            dp[i], came_from[i] = a, i - 1
        else:
            dp[i], came_from[i] = b2, i - 2
        print(f"{i:>2} | {dp[i-1]:>4}+{cost[i-1]:<3}={a:<4}| {dp[i-2]:>4}+{cost[i-2]:<3}={b2:<4}| {dp[i]}")
    path, pos = [], n
    while pos is not None and pos >= 2:
        path.append(came_from[pos])
        pos = came_from[pos]
    path.reverse()
    print("Các bậc phải trả phí:", path, "-> tổng", sum(cost[p] for p in path))
    return dp[n]

trace_min_cost([1, 100, 1, 1, 1, 100, 1, 1, 100, 1])

 i |       từ i-1 |       từ i-2 | dp[i]
 0 |            — |            — | 0
 1 |            — |            — | 0
 2 |    0+100=100 |    0+1  =1   | 1
 3 |    1+1  =2   |    0+100=100 | 2
 4 |    2+1  =3   |    1+1  =2   | 2
 5 |    2+1  =3   |    2+1  =3   | 3
 6 |    3+100=103 |    2+1  =3   | 3
 7 |    3+1  =4   |    3+100=103 | 4
 8 |    4+1  =5   |    3+1  =4   | 4
 9 |    4+100=104 |    4+1  =5   | 5
10 |    5+1  =6   |    4+100=104 | 6
Các bậc phải trả phí: [0, 2, 4, 6, 7, 9] -> tổng 6


6

Đường đi in ra `[0, 2, 4, 6, 7, 9]` khớp đúng giải thích của đề.

Lưu ý: truy vết dừng khi `pos < 2` — tức là khi đã về vị trí xuất phát `0` hoặc `1` (miễn phí đứng vào).

## 8. Cách 4 — Tối ưu bộ nhớ: chỉ giữ 2 giá trị (lời giải chính)

Nhìn công thức: `dp[i]` chỉ cần `dp[i-1]` và `dp[i-2]`. Không cần giữ cả mảng — dùng **2 biến trượt**:

- `prev2` = `dp[i-2]`, `prev1` = `dp[i-1]`.
- Tính `cur = min(prev1 + cost[i-1], prev2 + cost[i-2])`.
- Trượt cửa sổ: `prev2, prev1 = prev1, cur`.

Phép gán `a, b = b, c` của Python tính **toàn bộ vế phải trước** rồi mới gán, nên không cần biến tạm.

In [11]:
a, b_ = 1, 2
a, b_ = b_, a + b_       # vế phải (2, 3) được tính trước
print(a, b_)

2 3


In [12]:
def min_cost_two_vars(cost):
    prev2, prev1 = 0, 0                     # dp[0], dp[1]
    for i in range(2, len(cost) + 1):
        cur = min(prev1 + cost[i - 1], prev2 + cost[i - 2])
        prev2, prev1 = prev1, cur
    return prev1

print(min_cost_two_vars([10, 15, 20]))
print(min_cost_two_vars([1, 100, 1, 1, 1, 100, 1, 1, 100, 1]))

15
6


### Cách 4b — Theo hint của LeetCode: đi ngược từ đỉnh

Hint định nghĩa khác: `g(i)` = chi phí nhỏ nhất **từ bậc `i` lên đỉnh** (đã tính cả `cost[i]`). Khi đó `g(n) = 0`, `g(n+1) = 0` (quy ước), `g(i) = cost[i] + min(g(i+1), g(i+2))`, đáp án `min(g(0), g(1))`.

Hai định nghĩa đều đúng — chọn cái nào **dễ giải thích** cho bạn. Ở đây ta còn có thể cập nhật **ngay trên `cost`** (không tốn bộ nhớ thêm) nếu được phép sửa input.

In [13]:
def min_cost_backward(cost):
    n = len(cost)
    g_next, g_next2 = 0, 0                  # g(i+1), g(i+2)
    for i in range(n - 1, -1, -1):          # đi lùi n-1, ..., 0
        g = cost[i] + min(g_next, g_next2)
        g_next2, g_next = g_next, g
    # sau vòng: g_next = g(0), g_next2 = g(1)
    return min(g_next, g_next2)

print(min_cost_backward([10, 15, 20]))
print(min_cost_backward([1, 100, 1, 1, 1, 100, 1, 1, 100, 1]))
print(list(range(4, -1, -1)))   # range lùi: start=4, stop=-1 (không lấy), step=-1

15
6
[4, 3, 2, 1, 0]


## 9. Code nộp LeetCode

### Bản 2 biến (khuyến nghị)

In [14]:
class Solution:
    def minCostClimbingStairs(self, cost: list[int]) -> int:
        prev2, prev1 = 0, 0
        for i in range(2, len(cost) + 1):
            prev2, prev1 = prev1, min(prev1 + cost[i - 1], prev2 + cost[i - 2])
        return prev1


sol = Solution()
print(sol.minCostClimbingStairs([10, 15, 20]))
print(sol.minCostClimbingStairs([1, 100, 1, 1, 1, 100, 1, 1, 100, 1]))

15
6


### Bản bảng `dp` và bản memo

Trên LeetCode đổi tên class thành `Solution`. Bản memo dùng `@cache`; LeetCode Python thường đã nâng giới hạn đệ quy, nhưng bản lặp an toàn hơn.

In [15]:
from functools import cache


class SolutionTable:
    def minCostClimbingStairs(self, cost: list[int]) -> int:
        n = len(cost)
        dp = [0] * (n + 1)
        for i in range(2, n + 1):
            dp[i] = min(dp[i - 1] + cost[i - 1], dp[i - 2] + cost[i - 2])
        return dp[n]


class SolutionMemo:
    def minCostClimbingStairs(self, cost: list[int]) -> int:
        @cache
        def f(i: int) -> int:
            if i <= 1:
                return 0
            return min(f(i - 1) + cost[i - 1], f(i - 2) + cost[i - 2])
        return f(len(cost))


print(SolutionTable().minCostClimbingStairs([10, 15, 20]), SolutionMemo().minCostClimbingStairs([10, 15, 20]))

15 15


## 10. Vì sao đúng?

**Mệnh đề:** với mọi `i`, `dp[i]` = chi phí nhỏ nhất để đứng ở vị trí `i`. Chứng minh quy nạp theo `i`:

- **Cơ sở:** `i = 0, 1` — được phép xuất phát ở đó miễn phí, không cách nào rẻ hơn 0 (mọi `cost >= 0`) → `dp = 0` đúng.
- **Bước quy nạp:** xét một cách đi **bất kỳ** tới `i >= 2`. Bước cuối phải từ `i-1` (trả `cost[i-1]`) hoặc `i-2` (trả `cost[i-2]`). Phần đường **trước** bước cuối là một cách tới `i-1` (hoặc `i-2`), nên tốn ít nhất `dp[i-1]` (hoặc `dp[i-2]`) theo giả thiết quy nạp. Vậy mọi cách đi tốn `>= min(dp[i-1] + cost[i-1], dp[i-2] + cost[i-2])`. Và giá trị đó **đạt được** bằng cách nối bước cuối vào đường tối ưu tương ứng. ⇒ `dp[i]` đúng.

Tính chất "đường tối ưu tới `i` chứa đường tối ưu tới bậc trước" gọi là **cấu trúc con tối ưu** (optimal substructure) — điều kiện để DP hoạt động. Cùng với **bài toán con lặp lại** (thấy ở Cách 1), đây là hai dấu hiệu nhận biết DP.

**Cách 4** chỉ là Cách 3 với bất biến: *đầu vòng `i`, `prev2 = dp[i-2]` và `prev1 = dp[i-1]`.*

## 11. Độ phức tạp

Gọi `n = len(cost)`.

| Cách | Time | Space phụ |
|---|---|---|
| 1. Đệ quy thuần | `O(φ^n)` ≈ `O(1.618^n)` | `O(n)` call stack |
| 2. Memo (top-down) | `O(n)` | `O(n)` cache + `O(n)` call stack |
| 3. Bảng `dp` | `O(n)` | `O(n)` |
| 4. Hai biến | `O(n)` | `O(1)` |

Mỗi `dp[i]` tính bằng một `min` hai số → hằng số công việc mỗi ô → `O(n)`.

In [16]:
import time, random

rng = random.Random(746)
big = [rng.randint(0, 999) for _ in range(1000)]
for name, f in [("bảng dp", min_cost_table), ("hai biến", min_cost_two_vars), ("đi ngược", min_cost_backward)]:
    t0 = time.perf_counter()
    r = f(big)
    print(f"{name:<9} -> {r}  ({(time.perf_counter() - t0) * 1000:.2f} ms)")

bảng dp   -> 206313  (0.29 ms)
hai biến  -> 206313  (0.14 ms)
đi ngược  -> 206313  (0.08 ms)


## 12. Bộ test

Ngoài ví dụ đề, ta viết thêm một **oracle** (lời giải chắc chắn đúng nhưng chậm): sinh **mọi** chuỗi bước nhảy hợp lệ bằng đệ quy liệt kê, lấy chi phí nhỏ nhất. Dùng nó để kiểm các cách nhanh trên mảng nhỏ.

In [17]:
def check(name, actual, expected):
    assert actual == expected, (name, actual, expected)
    print(f"PASS  {name}: {actual}")

sol = Solution()
check("đề 1", sol.minCostClimbingStairs([10, 15, 20]), 15)
check("đề 2", sol.minCostClimbingStairs([1, 100, 1, 1, 1, 100, 1, 1, 100, 1]), 6)
check("2 bậc", sol.minCostClimbingStairs([5, 3]), 3)
check("2 bậc bằng nhau", sol.minCostClimbingStairs([4, 4]), 4)
check("toàn 0", sol.minCostClimbingStairs([0, 0, 0, 0]), 0)
check("bẫy tham lam", sol.minCostClimbingStairs([0, 1, 2, 2]), 2)
check("bậc cuối rất đắt", sol.minCostClimbingStairs([1, 2, 999]), 2)

PASS  đề 1: 15
PASS  đề 2: 6
PASS  2 bậc: 3
PASS  2 bậc bằng nhau: 4
PASS  toàn 0: 0
PASS  bẫy tham lam: 2
PASS  bậc cuối rất đắt: 2


In [18]:
def oracle(cost):
    n = len(cost)
    best = float("inf")
    def walk(pos, spent):
        nonlocal best
        if pos >= n:            # nhảy tới n (đỉnh); pos > n không xảy ra vì chỉ nhảy +1/+2 từ <= n-1
            best = min(best, spent)
            return
        walk(pos + 1, spent + cost[pos])
        walk(pos + 2, spent + cost[pos])
    walk(0, 0)
    walk(1, 0)
    return best

impls = [min_cost_recursive, min_cost_memo, min_cost_table, min_cost_two_vars, min_cost_backward,
         Solution().minCostClimbingStairs, SolutionTable().minCostClimbingStairs, SolutionMemo().minCostClimbingStairs]
for _ in range(1500):
    cost = [rng.randint(0, 9) for _ in range(rng.randint(2, 11))]
    expect = oracle(cost)
    for f in impls:
        assert f(cost) == expect, (f, cost)
print("1500 mảng ngẫu nhiên: 8 cài đặt khớp oracle liệt kê mọi đường")

1500 mảng ngẫu nhiên: 8 cài đặt khớp oracle liệt kê mọi đường


`float("inf")` là **dương vô cùng** — mọi số đều nhỏ hơn nó, nên dùng làm giá trị khởi đầu khi tìm min.

Chú ý: khi nhảy +2 từ bậc `n-1`, ta tới `n+1` — vượt đỉnh. Đề coi "đỉnh" là vượt qua bậc cuối, nên `pos >= n` đều tính là đã lên đỉnh; nhảy +2 từ `n-1` không bao giờ rẻ hơn nhảy +1 (cùng phí `cost[n-1]`), nên không ảnh hưởng đáp án.

## 13. Lỗi người mới hay gặp

### Lỗi 1: Tham lam — luôn bước sang bậc rẻ hơn trong hai bậc kế tiếp

In [19]:
def wrong_greedy(cost):
    n = len(cost)
    pos = 0 if cost[0] <= cost[1] else 1
    total = 0
    while pos < n:
        total += cost[pos]
        if pos + 1 >= n or pos + 2 >= n:
            break
        pos = pos + 1 if cost[pos + 1] < cost[pos + 2] else pos + 2
    return total

case = [0, 1, 2, 2]
print("tham lam:", wrong_greedy(case), "| đúng:", min_cost_two_vars(case))

tham lam: 3 | đúng: 2


Tham lam chọn `0` (rẻ), rồi bậc 1 (phí 1 < 2), rồi phải trả tiếp bậc 2 hoặc 3 → tổng 3. Tối ưu thật là `0 → 2 → đỉnh` = 0 + 2 = 2. Lựa chọn rẻ **trước mắt** không đảm bảo rẻ **toàn cục** → cần DP.

### Lỗi 2: Trả `dp[n-1]` (nhầm đỉnh là bậc cuối)

In [20]:
def wrong_top(cost):
    n = len(cost)
    dp = [0] * (n + 1)
    for i in range(2, n + 1):
        dp[i] = min(dp[i - 1] + cost[i - 1], dp[i - 2] + cost[i - 2])
    return dp[n - 1]                      # SAI: đỉnh là n

print("sai :", wrong_top([10, 15, 20]), "| đúng:", min_cost_table([10, 15, 20]))

sai : 10 | đúng: 15


### Lỗi 3: Cộng phí của bậc **đích** thay vì bậc **xuất phát**

Viết `dp[i] = min(dp[i-1], dp[i-2]) + cost[i]` là dùng định nghĩa "chi phí **kể cả** bậc `i`". Định nghĩa này cũng dùng được — nhưng khi đó đáp án phải là `min(dp[n-1], dp[n-2])` và base case là `dp[0] = cost[0]`, `dp[1] = cost[1]`. Trộn lẫn hai định nghĩa là lỗi phổ biến.

In [21]:
def wrong_mixed(cost):
    n = len(cost)
    dp = [0] * (n + 1)
    for i in range(2, n):
        dp[i] = min(dp[i - 1], dp[i - 2]) + cost[i]     # định nghĩa "kể cả bậc i"
    return dp[n - 1]                                    # ...nhưng base case & đáp án theo định nghĩa khác

def right_inclusive(cost):
    n = len(cost)
    dp = [0] * n
    dp[0], dp[1] = cost[0], cost[1]
    for i in range(2, n):
        dp[i] = min(dp[i - 1], dp[i - 2]) + cost[i]
    return min(dp[n - 1], dp[n - 2])

c = [1, 100, 1, 1, 1, 100, 1, 1, 100, 1]
print("trộn định nghĩa:", wrong_mixed(c), "| định nghĩa 'kể cả' làm đúng:", right_inclusive(c), "| đáp án:", min_cost_two_vars(c))

trộn định nghĩa: 5 | định nghĩa 'kể cả' làm đúng: 6 | đáp án: 6


**Bài học:** viết ra **bằng lời** `dp[i]` nghĩa là gì trước khi code, rồi suy base case và đáp án **từ định nghĩa đó**.

### Lỗi 4: Trượt biến sai thứ tự

`prev1 = cur; prev2 = prev1` → `prev2` nhận giá trị **mới** thay vì cũ. Dùng gán đồng thời `prev2, prev1 = prev1, cur`.

In [22]:
def wrong_slide(cost):
    prev2, prev1 = 0, 0
    for i in range(2, len(cost) + 1):
        cur = min(prev1 + cost[i - 1], prev2 + cost[i - 2])
        prev1 = cur
        prev2 = prev1          # SAI: prev1 đã bị ghi đè
    return prev1

print("sai :", wrong_slide(c), "| đúng:", min_cost_two_vars(c))

sai : 9 | đúng: 6


## 14. Mẹo nhận dạng pattern và biến thể

- "Tối thiểu/tối đa chi phí", "bao nhiêu cách", mỗi bước có **vài lựa chọn**, lựa chọn sau phụ thuộc vị trí hiện tại → nghĩ **DP**.
- Hai dấu hiệu: **bài toán con lặp lại** (đệ quy gọi trùng) + **cấu trúc con tối ưu**.
- Quy trình 4 bước: (1) định nghĩa `dp[i]` bằng lời; (2) công thức truy hồi từ "bước cuối"; (3) base case; (4) thứ tự tính + đáp án ở đâu.
- Truy hồi chỉ dùng `k` ô trước → tối ưu bộ nhớ còn `k` biến.
- Bài cùng họ: 70 Climbing Stairs, 509 Fibonacci Number, 1137 N-th Tribonacci, 198 House Robber, 213 House Robber II, 1646, 2140 Solving Questions With Brainpower.

### 14.1. Biến thể LeetCode 70 — Climbing Stairs: đếm **số cách** lên `n` bậc (bước 1 hoặc 2)

Cùng câu hỏi "bước cuối từ đâu", nhưng thay `min(... + cost)` bằng **tổng** số cách: `ways(i) = ways(i-1) + ways(i-2)`.

In [23]:
def climb_ways(n):
    a, b2 = 1, 1          # ways(0), ways(1)
    for _ in range(2, n + 1):
        a, b2 = b2, a + b2
    return b2

print([climb_ways(n) for n in range(1, 8)])   # 1 2 3 5 8 13 21

[1, 2, 3, 5, 8, 13, 21]


### 14.2. Biến thể — Được nhảy 1, 2 **hoặc 3** bậc

Chỉ thêm một nhánh vào `min`. Base case: `dp[0] = dp[1] = 0` và nhảy được từ `i-3` khi `i >= 3`. (Vẫn chỉ được xuất phát ở 0 hoặc 1.)

In [24]:
def min_cost_jump3(cost):
    n = len(cost)
    dp = [0] * (n + 1)
    for i in range(2, n + 1):
        options = [dp[i - 1] + cost[i - 1], dp[i - 2] + cost[i - 2]]
        if i >= 3:
            options.append(dp[i - 3] + cost[i - 3])
        dp[i] = min(options)
    return dp[n]

print(min_cost_jump3([10, 15, 20]), min_cost_jump3([1, 100, 1, 1, 1, 100, 1, 1, 100, 1]))

10 4


### 14.3. Biến thể — House Robber (198) — cùng khung "hai biến trượt"

Không được lấy hai nhà kề nhau, tối đa tổng tiền: `best(i) = max(best(i-1), best(i-2) + nums[i])`.

In [25]:
def rob(nums):
    prev2, prev1 = 0, 0
    for x in nums:
        prev2, prev1 = prev1, max(prev1, prev2 + x)
    return prev1

print(rob([1, 2, 3, 1]), rob([2, 7, 9, 3, 1]))   # 4 12

4 12


## 15. Bài tập

### Bài 1 — Điền bảng bằng tay
Với `cost = [3, 2, 5, 1, 4]`, điền `dp[0..5]` theo Cách 3. Đáp án là bao nhiêu? Các bậc nào được trả phí?

### Bài 2 — Điền chỗ trống bản hai biến

In [26]:
def exercise_min_cost(cost: list[int]) -> int:
    prev2, prev1 = 0, 0
    for i in range(2, len(cost) + 1):
        # TODO 1: tính cur theo công thức truy hồi
        # TODO 2: trượt hai biến (gán đồng thời)
        pass
    return prev1

# assert exercise_min_cost([10, 15, 20]) == 15
# assert exercise_min_cost([1, 100, 1, 1, 1, 100, 1, 1, 100, 1]) == 6

### Bài 3 — Trả cả đường đi
Viết `min_cost_with_path(cost)` trả `(chi_phí, danh_sách_bậc_trả_phí)`. Nếu có nhiều đường tối ưu, trả đường bất kỳ (test chỉ kiểm tổng và tính hợp lệ).

In [27]:
def min_cost_with_path(cost: list[int]) -> tuple[int, list[int]]:
    # TODO
    pass

# total, path = min_cost_with_path([1, 100, 1, 1, 1, 100, 1, 1, 100, 1])
# assert total == 6 and sum([1, 100, 1, 1, 1, 100, 1, 1, 100, 1][p] for p in path) == 6

### Bài 4 — Climbing Stairs có bậc hỏng
Đếm số cách lên vị trí `n` (bắt đầu ở 0, mỗi bước +1 hoặc +2), biết các bậc trong tập `broken` **không được đặt chân**.

In [28]:
def climb_with_broken(n: int, broken: set[int]) -> int:
    # TODO
    pass

# assert climb_with_broken(4, set()) == 5
# assert climb_with_broken(4, {2}) == 1        # 0 -> 1 -> 3 -> 4
# assert climb_with_broken(3, {1, 2}) == 0

### Bài 5 — Bước nhảy tối đa `k`
Tổng quát Cách 3: được nhảy `1..k` bậc, xuất phát ở bất kỳ bậc nào trong `0..k-1`. Tìm chi phí nhỏ nhất lên đỉnh `n`.

In [29]:
def min_cost_k_jumps(cost: list[int], k: int) -> int:
    # TODO
    pass

# assert min_cost_k_jumps([10, 15, 20], 2) == 15
# assert min_cost_k_jumps([10, 15, 20], 3) == 10
# assert min_cost_k_jumps([5, 1, 5, 1, 5], 2) == 2
# assert min_cost_k_jumps([1, 9, 9, 1, 9, 9], 3) == 2

### Bài 6 — Mini challenge: House Robber II (213)
Các nhà xếp **thành vòng tròn** (nhà đầu và nhà cuối kề nhau). Tìm tổng tiền tối đa không lấy hai nhà kề.

In [30]:
def rob_circle(nums: list[int]) -> int:
    # TODO
    pass

# assert rob_circle([2, 3, 2]) == 3
# assert rob_circle([1, 2, 3, 1]) == 4
# assert rob_circle([5]) == 5

## 16. Gợi ý 3 mức

| Bài | Mức 1 — hướng nghĩ | Mức 2 — cấu trúc | Mức 3 — gần code |
|---|---|---|---|
| 1 | Dùng đúng công thức Cách 3 | Tính `dp[2]..dp[5]` | `dp[2] = min(0+2, 0+3) = 2` ... |
| 2 | Hai biến = hai ô cuối của bảng | `cur = min(...)` | `prev2, prev1 = prev1, cur` |
| 3 | Lưu "đến từ đâu" | Mảng `came_from` như trace | Lần ngược từ `n` tới khi `< 2`, rồi `reverse` |
| 4 | Bậc hỏng có 0 cách | `ways[i] = 0 if i in broken` | `ways[i] = ways[i-1] + ways[i-2]` cho bậc không hỏng |
| 5 | `dp[i] = 0` với `i < k` | `dp[i] = min(dp[i-j] + cost[i-j] for j in 1..k)` | chỉ xét `j` với `i - j >= 0` |
| 6 | Không thể lấy cả đầu lẫn cuối | Hai bài thẳng: bỏ nhà đầu / bỏ nhà cuối | `max(rob(nums[1:]), rob(nums[:-1]))`, xử lý `len == 1` |

## 17. Đáp án tham khảo

**Bài 1:** `cost = [3, 2, 5, 1, 4]`:
`dp[0]=0, dp[1]=0, dp[2]=min(0+2, 0+3)=2, dp[3]=min(2+5, 0+2)=2, dp[4]=min(2+1, 2+5)=3, dp[5]=min(3+4, 2+1)=3`. Đáp án `3`, trả phí các bậc `1` và `3` (`2 + 1`).

In [31]:
print("Bài 1:", trace_min_cost([3, 2, 5, 1, 4]))


def answer_min_cost(cost):
    prev2, prev1 = 0, 0
    for i in range(2, len(cost) + 1):
        cur = min(prev1 + cost[i - 1], prev2 + cost[i - 2])
        prev2, prev1 = prev1, cur
    return prev1


def answer_with_path(cost):
    n = len(cost)
    dp = [0] * (n + 1)
    came = [None] * (n + 1)
    for i in range(2, n + 1):
        a, b2 = dp[i - 1] + cost[i - 1], dp[i - 2] + cost[i - 2]
        dp[i], came[i] = (a, i - 1) if a <= b2 else (b2, i - 2)
    path, pos = [], n
    while pos >= 2:
        pos = came[pos]
        path.append(pos)
    return dp[n], path[::-1]


def answer_climb_broken(n, broken):
    ways = [0] * (n + 1)
    ways[0] = 1
    for i in range(1, n + 1):
        if i in broken:
            continue
        ways[i] = ways[i - 1] + (ways[i - 2] if i >= 2 else 0)
    return ways[n]


def answer_k_jumps(cost, k):
    n = len(cost)
    dp = [0] * (n + 1)
    for i in range(k, n + 1):
        dp[i] = min(dp[i - j] + cost[i - j] for j in range(1, k + 1) if i - j >= 0)
    return dp[n]


def answer_rob_circle(nums):
    if len(nums) == 1:
        return nums[0]
    return max(rob(nums[1:]), rob(nums[:-1]))


assert answer_min_cost([10, 15, 20]) == 15
assert answer_min_cost([1, 100, 1, 1, 1, 100, 1, 1, 100, 1]) == 6
c = [1, 100, 1, 1, 1, 100, 1, 1, 100, 1]
total, path = answer_with_path(c)
assert total == 6 and sum(c[p] for p in path) == 6
print("Bài 3:", total, path)
assert answer_climb_broken(4, set()) == 5
assert answer_climb_broken(4, {2}) == 1
assert answer_climb_broken(3, {1, 2}) == 0
assert answer_k_jumps([10, 15, 20], 2) == 15
assert answer_k_jumps([10, 15, 20], 3) == 10
assert answer_k_jumps([5, 1, 5, 1, 5], 2) == 2
assert answer_k_jumps([1, 9, 9, 1, 9, 9], 3) == 2
assert answer_k_jumps([10, 15, 20], 2) == answer_min_cost([10, 15, 20])
assert answer_rob_circle([2, 3, 2]) == 3
assert answer_rob_circle([1, 2, 3, 1]) == 4
assert answer_rob_circle([5]) == 5
print("Tất cả đáp án PASS")

 i |       từ i-1 |       từ i-2 | dp[i]
 0 |            — |            — | 0
 1 |            — |            — | 0
 2 |    0+2  =2   |    0+3  =3   | 2
 3 |    2+5  =7   |    0+2  =2   | 2
 4 |    2+1  =3   |    2+5  =7   | 3
 5 |    3+4  =7   |    2+1  =3   | 3
Các bậc phải trả phí: [1, 3] -> tổng 3
Bài 1: 3
Bài 3: 6 [0, 2, 4, 6, 7, 9]
Tất cả đáp án PASS


Ghi chú bài 5: quy ước "được **đứng** miễn phí ở bậc `0..k-1`" → `dp[i] = 0` với `i < k`, vòng lặp bắt đầu từ `i = k`. Với `k = 2` ta có lại đúng bài gốc. Ví dụ `[10, 15, 20]`, `k = 3`: đứng ở 0, trả 10, nhảy 3 bậc tới đỉnh → `10`. Nếu `n < k` thì đỉnh nằm ngay trong vùng xuất phát → `0`.

Ghi chú bài 6: chia vòng tròn thành **hai bài đường thẳng** (bỏ nhà đầu / bỏ nhà cuối) rồi lấy max — mẹo "phá vòng" rất hay gặp.

## 18. Template nhớ nhanh + Checklist submit

```python
class Solution:
    def minCostClimbingStairs(self, cost):
        prev2, prev1 = 0, 0                    # dp[0], dp[1]: đứng ở 0 hoặc 1 miễn phí
        for i in range(2, len(cost) + 1):      # tới tận đỉnh n
            prev2, prev1 = prev1, min(prev1 + cost[i - 1],   # bước cuối từ i-1
                                      prev2 + cost[i - 2])   # bước cuối từ i-2
        return prev1                           # dp[n]
```

**Câu thần chú:** *"dp[i] = rẻ nhất để ĐỨNG ở i — bước cuối từ i-1 hay i-2 — trả phí bậc rời đi — đáp án ở n."*

**Checklist trước khi submit:**
- [ ] Đã viết bằng lời `dp[i]` nghĩa là gì, và base case/đáp án suy ra **từ đúng định nghĩa đó**?
- [ ] Đỉnh là vị trí `n` (không phải `n - 1`) → vòng lặp tới `len(cost) + 1`?
- [ ] Phí cộng là phí của bậc **xuất phát** (`cost[i-1]`, `cost[i-2]`)?
- [ ] Trượt biến bằng gán đồng thời `prev2, prev1 = prev1, cur`?
- [ ] Không dùng tham lam; đã test phản ví dụ `[0, 1, 2, 2]` → `2`?
- [ ] Đã test `len(cost) == 2` và mảng toàn 0?